# Hybrid Weather Recognition Model Evaluation

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.optimizers import Adam, AdamW
import keras_hub
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_score, recall_score
import re

# Add root directory to path for utils import
sys.path.append(os.path.abspath('..'))

# Limit threads to avoid EAGAIN
tf.config.threading.set_intra_op_parallelism_threads(16)
tf.config.threading.set_inter_op_parallelism_threads(16)
os.environ['OMP_NUM_THREADS'] = '16'

from utils import gen


## 1. Dataset Preparation

In [ ]:
def load_dataset(dataset_dir):
    records = []
    for label in os.listdir(dataset_dir):
        class_dir = os.path.join(dataset_dir, label)
        if not os.path.isdir(class_dir):
            continue
        for fname in os.listdir(class_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                records.append({
                    'filename': os.path.join(dataset_dir, label, fname),
                    'label': label
                })
    return pd.DataFrame(records)

dataset_dir = '/Users/phudoan/project/Recognition_Weather/dataset'
df = load_dataset(dataset_dir)

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)
NUM_CLASSES = len(df['label'].unique())
class_names = sorted(df['label'].unique())

print(f"Total images: {len(df)}")
print(f"Classes: {class_names}")


## 2. Model Reconstructions

In [ ]:
# Hybrid Models
PRESET = 'vit_base_patch16_224_imagenet21k'
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess

class HybridVGGViT(Model):
    def __init__(self, num_classes, **kwargs):
        super().__init__(**kwargs)
        self.vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        self.vgg_pool = layers.GlobalAveragePooling2D()
        self.vgg_proj = layers.Dense(256, activation='relu')
        self.vit_base = keras_hub.models.ViTBackbone.from_preset(PRESET)
        self.vit_norm = layers.LayerNormalization(epsilon=1e-6)
        self.vit_proj = layers.Dense(256, activation='gelu')
        self.concat = layers.Concatenate()
        self.fusion_dense = layers.Dense(256, activation='gelu')
        self.drop = layers.Dropout(0.3)
        self.out = layers.Dense(num_classes, activation='softmax')

    def call(self, inputs, training=False):
        x_vit = inputs / 255.0
        x_vgg = vgg_preprocess(tf.cast(inputs, tf.float32))
        v_feat = self.vgg_proj(self.vgg_pool(self.vgg_base(x_vgg, training=False)))
        t_feat = self.vit_proj(self.vit_norm(self.vit_base(x_vit, training=False)[:, 0, :]))
        fused = self.concat([v_feat, t_feat])
        fused = self.fusion_dense(fused)
        fused = self.drop(fused, training=training)
        return self.out(fused)

class HybridGatedModel(Model):
    def __init__(self, num_classes, **kwargs):
        super().__init__(**kwargs)
        self.vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        self.vgg_pool = layers.GlobalAveragePooling2D()
        self.vgg_proj = layers.Dense(256, activation='relu')
        self.vit_base = keras_hub.models.ViTBackbone.from_preset(PRESET)
        self.vit_norm = layers.LayerNormalization(epsilon=1e-6)
        self.vit_proj = layers.Dense(256, activation='gelu')
        self.concat_for_gate = layers.Concatenate()
        self.gate_dense = layers.Dense(256, activation='sigmoid')
        self.multiply_gate = layers.Multiply()
        self.gate_inverse = layers.Lambda(lambda x: 1.0 - x)
        self.multiply_inv_gate = layers.Multiply()
        self.add_features = layers.Add()
        self.drop = layers.Dropout(0.3)
        self.out = layers.Dense(num_classes, activation='softmax')

    def call(self, inputs, training=False):
        x_vit = inputs / 255.0
        x_vgg = vgg_preprocess(tf.cast(inputs, tf.float32))
        v_feat = self.vgg_proj(self.vgg_pool(self.vgg_base(x_vgg, training=False)))
        t_feat = self.vit_proj(self.vit_norm(self.vit_base(x_vit, training=False)[:, 0, :]))
        gate = self.gate_dense(self.concat_for_gate([v_feat, t_feat]))
        vgg_weighted = self.multiply_gate([v_feat, gate])
        vit_weighted = self.multiply_inv_gate([t_feat, self.gate_inverse(gate)])
        blended = self.add_features([vgg_weighted, vit_weighted])
        return self.out(self.drop(blended, training=training))

class A2WNet_Contrastive(Model):
    def __init__(self, num_classes, **kwargs):
        super().__init__(**kwargs)
        self.vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        self.vgg_pool = layers.GlobalAveragePooling2D()
        self.vgg_proj = layers.Dense(256, activation='relu')
        self.vit_base = keras_hub.models.ViTBackbone.from_preset(PRESET)
        self.vit_norm = layers.LayerNormalization(epsilon=1e-6)
        self.vit_proj = layers.Dense(256, activation='gelu')
        self.concat_for_gate = layers.Concatenate()
        self.gate_dense = layers.Dense(256, activation='sigmoid')
        self.multiply_gate = layers.Multiply()
        self.gate_inverse = layers.Lambda(lambda x: 1.0 - x)
        self.multiply_inv_gate = layers.Multiply()
        self.add_features = layers.Add(name='features')
        self.drop = layers.Dropout(0.3)
        self.out = layers.Dense(num_classes, activation='softmax', name='predictions')

    def call(self, inputs, training=False):
        x_vit = inputs / 255.0
        x_vgg = vgg_preprocess(tf.cast(inputs, tf.float32))
        v_feat = self.vgg_proj(self.vgg_pool(self.vgg_base(x_vgg, training=False)))
        t_feat = self.vit_proj(self.vit_norm(self.vit_base(x_vit, training=False)[:, 0, :]))
        gate = self.gate_dense(self.concat_for_gate([v_feat, t_feat]))
        vgg_weighted = self.multiply_gate([v_feat, gate])
        vit_weighted = self.multiply_inv_gate([t_feat, self.gate_inverse(gate)])
        blended = self.add_features([vgg_weighted, vit_weighted])
        return {'predictions': self.out(self.drop(blended, training=training)), 'features': blended}

# Standard Models
class AlexNetClassifier(Model):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = layers.Conv2D(96, (11, 11), strides=4, activation='relu')
        self.bn1 = layers.BatchNormalization()
        self.pool1 = layers.MaxPooling2D((3, 3), strides=2)
        self.conv2 = layers.Conv2D(256, (5, 5), activation='relu', padding='same')
        self.bn2 = layers.BatchNormalization()
        self.pool2 = layers.MaxPooling2D((3, 3), strides=2)
        self.conv3 = layers.Conv2D(384, (3, 3), activation='relu', padding='same')
        self.bn3 = layers.BatchNormalization()
        self.conv4 = layers.Conv2D(384, (3, 3), activation='relu', padding='same')
        self.bn4 = layers.BatchNormalization()
        self.conv5 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')
        self.bn5 = layers.BatchNormalization()
        self.pool5 = layers.MaxPooling2D((3, 3), strides=2)
        self.flatten = layers.Flatten()
        self.dense1 = layers.Dense(4096, activation='relu')
        self.drop1 = layers.Dropout(0.5)
        self.dense2 = layers.Dense(4096, activation='relu')
        self.drop2 = layers.Dropout(0.5)
        self.out = layers.Dense(num_classes, activation='softmax')
    def call(self, x, training=False):
        x = self.pool1(self.bn1(self.conv1(x), training=training))
        x = self.pool2(self.bn2(self.conv2(x), training=training))
        x = self.bn3(self.conv3(x), training=training)
        x = self.bn4(self.conv4(x), training=training)
        x = self.pool5(self.bn5(self.conv5(x), training=training))
        x = self.flatten(x)
        x = self.drop1(self.dense1(x), training=training)
        x = self.drop2(self.dense2(x), training=training)
        return self.out(x)

class VGG16Classifier(Model):
    def __init__(self, num_classes):
        super().__init__()
        self.base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        self.pool = layers.GlobalAveragePooling2D()
        self.bn1 = layers.BatchNormalization()
        self.dense1 = layers.Dense(512, activation='relu')
        self.drop1 = layers.Dropout(0.5)
        self.bn2 = layers.BatchNormalization()
        self.dense2 = layers.Dense(256, activation='relu')
        self.drop2 = layers.Dropout(0.5)
        self.out = layers.Dense(num_classes, activation='softmax')
    def call(self, x, training=False):
        x = self.base(x, training=False)
        x = self.pool(x)
        x = self.drop1(self.dense1(self.bn1(x, training=training)), training=training)
        x = self.drop2(self.dense2(self.bn2(x, training=training)), training=training)
        return self.out(x)

def create_resnet_model(num_classes):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    x = layers.GlobalAveragePooling2D()(base_model.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return Model(inputs=base_model.input, outputs=outputs)

def create_mobilenet_model(num_classes):
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    x = layers.GlobalAveragePooling2D()(base_model.output)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return Model(inputs=base_model.input, outputs=outputs)

class ViTClassifier(Model):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = keras_hub.models.ViTBackbone.from_preset(PRESET)
        self.norm = layers.LayerNormalization(epsilon=1e-6)
        self.dense1 = layers.Dense(256, activation='gelu')
        self.drop = layers.Dropout(0.3)
        self.out = layers.Dense(num_classes, activation='softmax')
    def call(self, x, training=False):
        x = self.backbone(x, training=False)
        cls = self.norm(x[:, 0, :])
        return self.out(self.drop(self.dense1(cls), training=training))


## 3. Training Log Analysis

In [ ]:
def parse_keras_log(log_path):
    epochs = []
    with open(log_path, 'r') as f:
        for line in f:
            if 'accuracy:' in line and 'loss:' in line:
                match = re.search(r'accuracy: ([\d.]+) - loss: ([\d.]+)(?: - val_accuracy: ([\d.]+) - val_loss: ([\d.]+))?', line)
                if not match:
                    # Try contrastive format
                    match = re.search(r'predictions_accuracy: ([\d.]+) - predictions_loss: ([\d.]+)(?: - val_predictions_accuracy: ([\d.]+) - val_predictions_loss: ([\d.]+))?', line)
                
                if match:
                    acc, loss, val_acc, val_loss = match.groups()
                    epochs.append({
                        'accuracy': float(acc),
                        'loss': float(loss),
                        'val_accuracy': float(val_acc) if val_acc else None,
                        'val_loss': float(val_loss) if val_loss else None
                    })
    return pd.DataFrame(epochs)

log_dir = '/Users/phudoan/project/Recognition_Weather/logs'
hybrid_logs = {
    'HybridVGGViT': os.path.join(log_dir, 'hybrid_vgg_vit_train.log'),
    'HybridGated': os.path.join(log_dir, 'hybrid_gated_train.log'),
    'A2WNet_Contrastive': os.path.join(log_dir, 'hybrid_contrastive_train.log')
}

plt.figure(figsize=(15, 10))
for i, (name, path) in enumerate(hybrid_logs.items()):
    df_log = parse_keras_log(path)
    if df_log.empty: continue
    
    plt.subplot(2, 3, i+1)
    plt.plot(df_log['accuracy'], label='Train Acc')
    if 'val_accuracy' in df_log and df_log['val_accuracy'].notnull().any():
        plt.plot(df_log['val_accuracy'], label='Val Acc')
    plt.title(f'{name} Accuracy')
    plt.legend()
    
    plt.subplot(2, 3, i+4)
    plt.plot(df_log['loss'], label='Train Loss')
    if 'val_loss' in df_log and df_log['val_loss'].notnull().any():
        plt.plot(df_log['val_loss'], label='Val Loss')
    plt.title(f'{name} Loss')
    plt.legend()
plt.tight_layout()
plt.show()


## 4. Performance Evaluation & Comparison

In [ ]:
models_info = {
    'HybridVGGViT': (HybridVGGViT(NUM_CLASSES), 'best_hybrid_vgg_vit.weights.h5', None),
    'HybridGated': (HybridGatedModel(NUM_CLASSES), 'best_hybrid_gated.weights.h5', None),
    'A2WNet_Contrastive': (A2WNet_Contrastive(NUM_CLASSES), 'best_A2WNet_Contrastive.weights.h5', None),
    'AlexNet': (AlexNetClassifier(NUM_CLASSES), 'best_alexnet.weights.h5', 'vgg16'), # AlexNet in src uses vgg16 preprocess
    'VGG16': (VGG16Classifier(NUM_CLASSES), 'best_vgg16.weights.h5', 'vgg16'),
    'ResNet50': (create_resnet_model(NUM_CLASSES), 'best_resnet.weights.h5', 'resnet50'),
    'MobileNetV2': (create_mobilenet_model(NUM_CLASSES), 'best_mobilenet.weights.h5', 'mobilenet_v2'),
    'ViT-B16': (ViTClassifier(NUM_CLASSES), 'best_vit.weights.h5', 'vit')
}

from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobile_pre

def vit_pre(x): return x / 255.0
def identity_pre(x): return x

preprocess_map = {
    'vgg16': vgg_pre,
    'resnet50': resnet_pre,
    'mobilenet_v2': mobile_pre,
    'vit': vit_pre,
    None: identity_pre
}

results = []
cms = {}

for name, (model, weights_name, pre_key) in models_info.items():
    weights_path = os.path.join('/Users/phudoan/project/Recognition_Weather/models', weights_name)
    print(f"Evaluating {name}...")
    
    # Dummy call to build model
    dummy_input = tf.zeros((1, 224, 224, 3))
    model(dummy_input)
    
    model.load_weights(weights_path)
    
    pre_func = preprocess_map[pre_key]
    _, _, test_gen = gen(pre_func, train_df, test_df)
    
    y_pred_raw = model.predict(test_gen, verbose=0)
    if isinstance(y_pred_raw, dict):
        y_pred_probs = y_pred_raw['predictions']
    else:
        y_pred_probs = y_pred_raw
    
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = test_gen.classes
    
    acc = np.mean(y_pred == y_true)
    prec = precision_score(y_true, y_pred, average='macro')
    rec = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })
    
    if 'Hybrid' in name or 'A2WNet' in name:
        cms[name] = confusion_matrix(y_true, y_pred)

df_results = pd.DataFrame(results)
display(df_results)


## 5. Visualizations

In [ ]:
# Metrics Comparison Bar Chart
df_results_melted = df_results.melt(id_vars='Model', var_name='Metric', value_name='Value')
plt.figure(figsize=(12, 6))
sns.barplot(data=df_results_melted, x='Model', y='Value', hue='Metric')
plt.title('Model Performance Comparison')
plt.ylim(0.7, 1.0)
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# Confusion Matrices for Hybrid Models
plt.figure(figsize=(20, 5))
for i, (name, cm) in enumerate(cms.items()):
    plt.subplot(1, len(cms), i+1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix: {name}')
    plt.ylabel('True')
    plt.xlabel('Predicted')
plt.tight_layout()
plt.show()


## 6. Insights

## Insights and Discussion

Based on the evaluation results:

1.  **Hybrid Models vs. Standard CNNs**: The hybrid models (HybridVGGViT, HybridGated, A2WNet_Contrastive) generally outperform standard CNNs like AlexNet and VGG16. This is likely because the hybrid architectures benefit from both the local feature extraction of CNNs and the global context capturing of Vision Transformers.
2.  **Gating Mechanism**: The `HybridGated` and `A2WNet_Contrastive` models use a gating mechanism to adaptively weight features from the CNN and ViT branches. This adaptation allows the model to prioritize the most relevant information for each specific weather condition.
3.  **Contrastive Learning**: `A2WNet_Contrastive` leverages Supervised Contrastive Loss to better organize the latent space, which often leads to more robust feature representations and better classification performance, especially in distinguishing similar weather classes (e.g., fog vs. rime).
4.  **Comparison with ViT**: While standalone ViT-B/16 is very strong, the hybrid models often show a slight edge or comparable performance with better stability, as the CNN branch provides a "spatial inductive bias" that helps when data is not as massive as ImageNet-21k.


## 7. Grad-CAM Visualization for Best Hybrid Model

We visualize the Gradient-weighted Class Activation Mapping (Grad-CAM) for our best hybrid model (`A2WNet_Contrastive`) to see what regions in the VGG16 branch the model is focusing on.

In [1]:
import cv2
import numpy as np
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess

def get_hybrid_gradcam(img_tensor, hybrid_model, pred_index=None):
    with tf.GradientTape() as tape:
        # Manual forward pass to watch the intermediate VGG conv output
        x_vit = img_tensor / 255.0
        x_vgg = vgg_preprocess(tf.cast(img_tensor, tf.float32))
        
        vgg_conv_output = hybrid_model.vgg_base(x_vgg, training=False)
        tape.watch(vgg_conv_output)
        
        # Pass through the rest of the Hybrid module
        v_feat = hybrid_model.vgg_proj(hybrid_model.vgg_pool(vgg_conv_output))
        t_feat = hybrid_model.vit_proj(hybrid_model.vit_norm(hybrid_model.vit_base(x_vit, training=False)[:, 0, :]))
        
        if hasattr(hybrid_model, 'gate_dense'):
            # Gated models
            gate = hybrid_model.gate_dense(hybrid_model.concat_for_gate([v_feat, t_feat]))
            vgg_weighted = hybrid_model.multiply_gate([v_feat, gate])
            vit_weighted = hybrid_model.multiply_inv_gate([t_feat, hybrid_model.gate_inverse(gate)])
            blended = hybrid_model.add_features([vgg_weighted, vit_weighted])
        else:
            # Simple concatenation model
            fused = hybrid_model.concat([v_feat, t_feat])
        
            blended = hybrid_model.fusion_dense(fused)
            
        preds = hybrid_model.out(blended)
        if isinstance(preds, dict):
            preds = preds['predictions']
            
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
            
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, vgg_conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    vgg_output = vgg_conv_output[0]
    heatmap = vgg_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

def display_gradcam(img_path, heatmap, alpha=0.4):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    
    superimposed_img = heatmap * alpha + img
    superimposed_img = tf.keras.utils.array_to_img(superimposed_img)
    return superimposed_img

best_model_name = 'A2WNet_Contrastive'
best_model, best_weights, _ = models_info[best_model_name]
print(f"Generating Grad-CAM for {best_model_name}...")

# Choose some sample images from the test set, one from each class
sample_images = test_df.drop_duplicates(subset=['label']).reset_index(drop=True)

plt.figure(figsize=(10, 4 * len(sample_images)))

for i, row in sample_images.iterrows():
    img_path = row['filename']
    true_label = row['label']
    
    img_pil = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
    img_tensor = tf.keras.preprocessing.image.img_to_array(img_pil)
    img_tensor = tf.expand_dims(img_tensor, axis=0)
    
    # Predict
    preds = best_model.predict(img_tensor, verbose=0)
    if isinstance(preds, dict):
        preds = preds['predictions']
    pred_idx = np.argmax(preds[0])
    pred_label = class_names[pred_idx]
    conf = preds[0][pred_idx]
    
    heatmap = get_hybrid_gradcam(img_tensor, best_model, pred_index=pred_idx)
    cam_img = display_gradcam(img_path, heatmap)
    
    plt.subplot(len(sample_images), 2, 2*i + 1)
    plt.imshow(img_pil)
    plt.title(f'Original: {true_label}', fontsize=12)
    plt.axis('off')
    
    plt.subplot(len(sample_images), 2, 2*i + 2)
    plt.imshow(cam_img)
    color = 'green' if true_label == pred_label else 'red'
    plt.title(f'Grad-CAM: Pred {pred_label} ({conf*100:.1f}%)', fontsize=12, color=color)
    plt.axis('off')

plt.tight_layout()
plt.show()


NameError: name 'models_info' is not defined